In [1]:
!pip install -q -U transformers datasets accelerate peft sentence-transformers wandb 


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, gc, re, random, warnings
import numpy as np, pandas as pd, torch

In [ ]:
import wandb
try:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    print("W&B logged in via secret")
except Exception as e:
    print("Secret not found, will prompt interactively:", e)

In [ ]:

warnings.filterwarnings("ignore")

class CFG:
    # --- data ---
    DATA_DIR = "/kaggle/input/smart-mcq-solver-challenge"  # change if your dataset folder name differs
    OPTS     = ["A", "B", "C", "D", "E"]                   # the five answer choices
    SEED     = 42                                          # fixes randomness -> reproducible

    # --- score-driver model (DeBERTa + LoRA, Milestone 4) ---
    MODEL_NAME = "microsoft/deberta-v3-base"
    MAX_LEN    = 256      # max tokens per (question + one option) pair
    EPOCHS     = 3        # passes over the training data
    LR         = 2e-4     # LoRA uses a higher learning rate than full fine-tuning
    TRAIN_BS   = 4        # training batch size
    EVAL_BS    = 8        # evaluation batch size

    # --- pretrained zero-shot model (Milestone 2) ---
    ST_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"

    # --- Weights & Biases ---
    WANDB_PROJECT = "23f2002523-t22026"   # <-- put YOUR W&B project name here

def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_everything(CFG.SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("torch:", torch.__version__)

In [ ]:
train = pd.read_csv(f"/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv(f"/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
samp  = pd.read_csv(f"/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("train shape:", train.shape)   # (rows, columns)
print("test shape :", test.shape)
print("columns    :", list(train.columns))

# how often each letter is the correct answer (checks for imbalance)
print("\nAnswer counts:")
print(train["answer"].value_counts().sort_index())

# check for any missing/empty cells
print("\nMissing values -> train:", train.isnull().sum().sum(),
      "| test:", test.isnull().sum().sum())

# peek at one full example
print("\n--- Example row 0 ---")
print("PROMPT:", train.loc[0, "prompt"][:200])
for o in CFG.OPTS:
    print(f"{o}:", str(train.loc[0, o])[:120])
print("ANSWER:", train.loc[0, "answer"])

train.head(2)

In [ ]:
from sklearn.metrics import f1_score

def map_at_3(true_letters, pred_lists):
    """
    true_letters: list of correct letters, e.g. ["B", "A", ...]
    pred_lists:   list of ranked top-3 guesses, e.g. [["B","C","A"], ...]
    Returns the mean MAP@3 score.
    """
    total = 0.0
    for gt, preds in zip(true_letters, pred_lists):
        for rank, p in enumerate(preds[:3]):     # rank = 0,1,2
            if p == gt:
                total += 1.0 / (rank + 1)         # 1, 1/2, or 1/3
                break                             # stop at first match
    return total / len(true_letters)

def probs_to_top3(probs):
    """probs: array (N,5) of scores per option -> list of ranked top-3 letter lists."""
    order = np.argsort(-probs, axis=1)[:, :3]     # indices of 3 highest scores
    return [[CFG.OPTS[j] for j in row] for row in order]

def evaluate(true_letters, probs):
    """Returns map@3, accuracy, and macro-F1 in one dict (used for W&B comparison)."""
    top3   = probs_to_top3(probs)
    y_true = np.array([CFG.OPTS.index(a) for a in true_letters])
    y_pred = probs.argmax(1)
    return {
        "map@3":    round(map_at_3(true_letters, top3), 4),
        "accuracy": round(float((y_pred == y_true).mean()), 4),
        "macro_f1": round(float(f1_score(y_true, y_pred, average="macro")), 4),
    }

# quick self-test so we trust the function
test_true = ["A", "B"]
test_pred = [["A", "X", "Y"],   # correct in 1st place -> 1.0
             ["X", "B", "Y"]]   # correct in 2nd place -> 0.5
print("Self-test MAP@3 (should be 0.75):", map_at_3(test_true, test_pred))

In [ ]:
# templated wrappers to strip (case-insensitive)
PREFIX = re.compile(r"^\s*(pick the best possible answer|choose the correct.*?|"
                    r"select the (?:correct|best).*?|identify the.*?)\s*:\s*", re.I)
SUFFIX = re.compile(r"\s*(among the listed options|from the following choices|"
                    r"from the options|carefully)\.?\s*$", re.I)

def clean_text(s):
    s = str(s).strip()          # force string, trim spaces
    s = PREFIX.sub("", s)       # drop leading template
    s = SUFFIX.sub("", s)       # drop trailing template
    return re.sub(r"\s+", " ", s).strip()   # collapse extra spaces

# apply to both train and test; clean the prompt AND the options
for df in (train, test):
    df["prompt_clean"] = df["prompt"].map(clean_text)
    for o in CFG.OPTS:
        df[o] = df[o].map(clean_text)

# before / after check
print("BEFORE:", repr(train.loc[0, "prompt"]))
print("AFTER :", repr(train.loc[0, "prompt_clean"]))